# MethylGPT quickstart

Predict age and 25 full-array clock outputs, convert a 27K-array profile to the full 49,156-CpG vocabulary, and compute sample embeddings. The example file holds six public 450K profiles from GSE54848 (dermal fibroblasts and iPSCs after three passages); none of them were used to train MethylGPT.

Set `BUNDLE` to the unpacked model bundle (`methylgpt-1.1`). Predicted age is the mean of three training runs. A GPU is used when available; CPU also works.

In [1]:
import os
import numpy as np
import pandas as pd
from methylgpt import MethylGPT

BUNDLE = os.environ.get('METHYLGPT_BUNDLE', 'methylgpt-1.1')
model = MethylGPT.from_pretrained(BUNDLE)
print(model.device, len(model.vocab), 'vocabulary CpGs')

cpu 49156 vocabulary CpGs


## Input
Rows are samples, columns are CpG probe IDs, values are beta values in [0, 1]. Missing CpGs can be absent or NaN; no imputation is needed beforehand.

In [2]:
betas = pd.read_csv('GSE54848_example_betas.csv', index_col=0)
betas.iloc[:, :5]

,cg00000109,cg00000292,cg00002033,cg00002426,cg00002719
GSM1345997_Day0_1,0.6339,0.7218,0.0184,0.2035,0.0079
GSM1345998_Day0_2,0.5074,0.8183,0.0333,0.2094,0.0129
GSM1345999_Day0_3,0.6438,0.7353,0.0160,0.2330,NaN
GSM1346021_p3_1,0.8386,0.8646,0.0127,0.3590,0.0292
GSM1346022_p3_2,0.8497,0.8604,0.0154,0.3402,0.0165
GSM1346023_p3_3,0.7618,0.8359,0.0082,0.2741,0.0201


## Age, sex, tissue and distilled clocks

In [3]:
pred = model.predict(betas)
pred[['vocab_cpgs_observed', 'age', 'p_male', 'tissue', 'GrimAge2', 'DunedinPACE', 'PhenoAge']].round(2)

,vocab_cpgs_observed,age,p_male,tissue,GrimAge2,DunedinPACE,PhenoAge
GSM1345997_Day0_1,48939,19.55,0.00,Skin,65.37,2.17,31.75
GSM1345998_Day0_2,48972,18.62,0.00,Skin,63.33,2.18,28.44
GSM1345999_Day0_3,48945,20.49,0.00,Skin,65.56,2.16,30.56
GSM1346021_p3_1,48970,9.23,0.01,Cells,25.13,1.74,-51.31
GSM1346022_p3_2,48971,9.21,0.01,Cells,25.17,1.74,-53.13
GSM1346023_p3_3,48974,8.81,0.00,Cells,22.41,1.81,-43.73


## Array conversion: keep only CpGs on the 27K array, impute the rest
The 27K array covers about 45% of the vocabulary. Here the other CpGs are removed and then predicted, and the predictions are compared with the measured values.

In [4]:
on27k = set(pd.read_csv('hm27_vocab_cpgs.txt', header=None)[0])
reduced = betas[[c for c in betas.columns if c in on27k]]
full = model.impute(reduced)
held = [c for c in betas.columns if c not in on27k]
err = (full[held] - betas[held]).abs()
print('kept', reduced.shape[1], 'CpGs; predicted', len(held), 'measured CpGs; mean absolute error',
      round(float(np.nanmean(err.to_numpy())), 4))

kept 22231 CpGs; predicted 26768 measured CpGs; mean absolute error 0.0528


In [5]:
pd.DataFrame({'age, all CpGs': pred.age, 'age, 27K CpGs only': model.predict_age(reduced)}).round(1)

,"age, all CpGs","age, 27K CpGs only"
GSM1345997_Day0_1,19.6,19.0
GSM1345998_Day0_2,18.6,22.3
GSM1345999_Day0_3,20.5,17.7
GSM1346021_p3_1,9.2,10.1
GSM1346022_p3_2,9.2,9.9
GSM1346023_p3_3,8.8,10.9


## Reconstruction and residuals
`reconstruct()` returns the factor model's reconstruction of every vocabulary CpG, including the measured ones; `residuals()` returns the measured value minus this reconstruction (NaN where a CpG is not measured). `impute()`, used above, keeps measured values and fills only the others.

In [6]:
rec = model.reconstruct(betas)
res = model.residuals(betas)
print(rec.shape, res.shape)
print('mean absolute residual at measured CpGs:', round(float(np.nanmean(np.abs(res.to_numpy()))), 4))
print('unmeasured CpGs in the first profile:', int(res.iloc[0].isna().sum()))

(6, 49156) (6, 49156)
mean absolute residual at measured CpGs: 0.036
unmeasured CpGs in the first profile: 217


## Sample embeddings
A 2,048-dimensional representation (standardised factor scores plus pooled transformer latents) that is the input of the prediction heads; use it for downstream models, and `factor_scores()` (below) for clustering.

In [7]:
emb = model.embed(betas)
print(emb.shape)
np.round(np.corrcoef(emb.to_numpy())[:3, :3], 3)

(6, 2048)


array([[1.   , 0.964, 0.994],
       [0.964, 1.   , 0.964],
       [0.994, 0.964, 1.   ]])

## Factor scores and the features of the paper's disease models
`factor_scores()` gives the 1,536 factor scores (use these for clustering and nearest-neighbour search). `embed()` returns the embedding of the released encoder; the disease models of the paper were fitted on the embedding of an earlier checkpoint, returned by `embed(stage='age_stage')`, and `disease_features()` returns their complete feature set (that embedding, the 25 clock outputs in standardized units and the 49,156 factor-model residuals).

In [8]:
fs = model.factor_scores(betas)
E = model.embed(betas, stage='age_stage')
F = model.disease_features(betas)
print(fs.shape, E.shape, F.shape)
print('correlation of the two embeddings, first profile:', round(float(np.corrcoef(E.iloc[0], emb.iloc[0])[0, 1]), 3))

(6, 1536) (6, 2048) (6, 51229)
correlation of the two embeddings, first profile: 0.992
